# 🔬 LangGraph Research Workflow with Memory

## What We're Building

A research workflow that **accumulates findings over multiple sessions**:
1. **Define research topic** — set the question and scope
2. **Research a sub-topic** — LLM generates findings for a specific angle
3. **Accumulate** — new findings are appended to a persistent knowledge base
4. **Synthesize** — LLM produces a comprehensive summary from all accumulated findings

The key feature: **persistence**. You can invoke the workflow multiple times
on the same thread, each time exploring a different sub-topic, and the
knowledge base grows across invocations.

## Architecture

```
Session 1: "climate policy"     Session 2: "renewable energy"     Session 3: "carbon capture"
         │                                │                                │
         ▼                                ▼                                ▼
   ┌──────────┐                    ┌──────────┐                    ┌──────────┐
   │ Research  │                    │ Research  │                    │ Research  │
   │ Sub-topic │                    │ Sub-topic │                    │ Sub-topic │
   └────┬─────┘                    └────┬─────┘                    └────┬─────┘
        │                               │                               │
        ▼                               ▼                               ▼
   ┌──────────┐                    ┌──────────┐                    ┌──────────┐
   │ Accumulate│──── persists ────▶│ Accumulate│──── persists ────▶│ Accumulate│
   └────┬─────┘                    └────┬─────┘                    └────┬─────┘
        │                               │                               │
        ▼                               ▼                               ▼
   ┌──────────┐                    ┌──────────┐                    ┌──────────┐
   │ Synthesize│                    │ Synthesize│                    │ Synthesize│
   └──────────┘                    └──────────┘                    └──────────┘
```

## Stack
- **LangGraph** — workflow graph with state persistence
- **MemorySaver** — in-memory persistence across invocations
- **Ollama** — local LLM for research and synthesis

## Step 1 — Imports & Configuration

In [ ]:
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_ollama import ChatOllama
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3:4b", temperature=0.3)
print("All imports successful!")

## Step 2 — Define State with Accumulating Findings

The `findings_log` field uses `Annotated[list, operator.add]` so that
each node return **appends** to the list instead of replacing it.
Combined with `MemorySaver`, this gives us cross-session accumulation.

In [ ]:
class ResearchState(TypedDict):
    main_topic: str                                     # Overall research question
    current_subtopic: str                               # Sub-topic for this session
    latest_findings: str                                # Findings from current session
    findings_log: Annotated[list[str], operator.add]    # Accumulated across sessions
    synthesis: str                                       # Combined summary
    session_count: int                                   # How many sessions so far

print("ResearchState defined (findings_log accumulates via operator.add)")

## Step 3 — Define Graph Nodes

| Node | Role |
|------|------|
| `research_subtopic` | LLM researches the current sub-topic |
| `accumulate` | Appends findings to the persistent log |
| `synthesize` | LLM synthesizes all accumulated findings |

In [ ]:
research_prompt = ChatPromptTemplate.from_template(
    """You are a research assistant. Investigate this sub-topic thoroughly.

Main Research Topic: {main_topic}
Current Sub-topic: {subtopic}

Previously accumulated findings (for context — do not repeat these):
{prior_findings}

Provide:
- KEY FACTS (3-5 bullet points)
- IMPORTANT NUANCES or debates
- CONNECTIONS to the main topic
- OPEN QUESTIONS worth investigating next

Be specific and cite concepts, not generic statements."""
)


def research_subtopic(state: ResearchState) -> dict:
    """Research the current sub-topic using LLM."""
    session = state.get("session_count", 0) + 1
    print(f"🔍 Node: research_subtopic — Session #{session}: '{state['current_subtopic']}'")

    prior = state.get("findings_log", [])
    prior_text = "\n---\n".join(prior) if prior else "(none yet)"

    chain = research_prompt | llm | StrOutputParser()
    findings = chain.invoke({
        "main_topic": state["main_topic"],
        "subtopic": state["current_subtopic"],
        "prior_findings": prior_text,
    })
    print(f"     Generated {len(findings)} chars of findings")
    return {"latest_findings": findings, "session_count": session}


def accumulate(state: ResearchState) -> dict:
    """Append current findings to the persistent log."""
    subtopic = state["current_subtopic"]
    entry = f"[Sub-topic: {subtopic}]\n{state['latest_findings']}"
    total = len(state.get("findings_log", [])) + 1
    print(f"📚 Node: accumulate — Now {total} entries in knowledge base")
    return {"findings_log": [entry]}


synthesize_prompt = ChatPromptTemplate.from_template(
    """You are a senior researcher. Synthesize all accumulated findings
into a coherent research summary.

Main Topic: {main_topic}

All Accumulated Findings:
{all_findings}

Write a structured research summary with:
- EXECUTIVE SUMMARY (2-3 sentences)
- KEY THEMES across all sub-topics
- CONNECTIONS AND PATTERNS between findings
- GAPS — what sub-topics should be explored next?
- PRELIMINARY CONCLUSIONS

The summary should feel like a literature review section."""
)


def synthesize(state: ResearchState) -> dict:
    """Synthesize all accumulated findings into a summary."""
    all_findings = state.get("findings_log", [])
    print(f"📝 Node: synthesize — Combining {len(all_findings)} entries...")
    chain = synthesize_prompt | llm | StrOutputParser()
    summary = chain.invoke({
        "main_topic": state["main_topic"],
        "all_findings": "\n---\n".join(all_findings),
    })
    return {"synthesis": summary}


print("All nodes defined")

## Step 4 — Build Graph with Persistence

In [ ]:
workflow = StateGraph(ResearchState)

workflow.add_node("research_subtopic", research_subtopic)
workflow.add_node("accumulate", accumulate)
workflow.add_node("synthesize", synthesize)

workflow.set_entry_point("research_subtopic")
workflow.add_edge("research_subtopic", "accumulate")
workflow.add_edge("accumulate", "synthesize")
workflow.add_edge("synthesize", END)

memory = MemorySaver()
app = workflow.compile(checkpointer=memory)
print("Research workflow compiled with persistence")

## Step 5 — Session 1: Research First Sub-topic

In [ ]:
config = {"configurable": {"thread_id": "research-climate"}}

result1 = app.invoke({
    "main_topic": "Impact of climate change on global food security",
    "current_subtopic": "crop yield projections under warming scenarios",
    "latest_findings": "",
    "findings_log": [],
    "synthesis": "",
    "session_count": 0,
}, config)

print("\n" + "="*60)
print(f"📊 SESSION 1 FINDINGS ({result1['session_count']} session total)")
print("="*60)
print(result1["latest_findings"][:500] + "...")
print(f"\nAccumulated entries: {len(result1['findings_log'])}")

## Step 6 — Session 2: Research Second Sub-topic

Notice: we reuse the same `thread_id` so the graph picks up
the previously accumulated findings from Session 1.

In [ ]:
result2 = app.invoke({
    "main_topic": "Impact of climate change on global food security",
    "current_subtopic": "water scarcity and irrigation challenges",
    "latest_findings": "",
    "synthesis": "",
}, config)

print("\n" + "="*60)
print(f"📊 SESSION 2 FINDINGS ({result2['session_count']} sessions total)")
print("="*60)
print(result2["latest_findings"][:500] + "...")
print(f"\nAccumulated entries: {len(result2['findings_log'])}")

## Step 7 — Session 3: Research Third Sub-topic & Full Synthesis

In [ ]:
result3 = app.invoke({
    "main_topic": "Impact of climate change on global food security",
    "current_subtopic": "supply chain disruption from extreme weather events",
    "latest_findings": "",
    "synthesis": "",
}, config)

print("\n" + "="*60)
print(f"📊 SESSION 3 — ACCUMULATED {len(result3['findings_log'])} ENTRIES")
print("="*60)
for i, entry in enumerate(result3["findings_log"]):
    header = entry.split("\n")[0]
    print(f"  {i+1}. {header}")

print("\n" + "="*60)
print("📝 COMPREHENSIVE SYNTHESIS (from all 3 sessions)")
print("="*60)
print(result3["synthesis"])

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Accumulating state** | `Annotated[list, operator.add]` appends, never replaces |
| **Cross-session memory** | Same `thread_id` + `MemorySaver` = persistent state |
| **Context-aware research** | Prior findings are passed to LLM to avoid repetition |
| **Progressive synthesis** | Summary improves as more sub-topics are explored |

## Why Persistence for Research?

- Research is iterative — you explore one angle at a time
- Accumulated context prevents redundant work
- Synthesis improves with each new sub-topic
- Full audit trail of what was explored and when

## 🔧 Extensions

- **Real search**: Replace LLM-only research with web search or document retrieval
- **Citation tracking**: Tag each finding with its source
- **Topic suggestion**: Have the LLM suggest the next sub-topic automatically
- **SQLite persistence**: Replace `MemorySaver` with `SqliteSaver` for durable storage